Parametrer le drive sur colab uniquement

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
BIGQUERY_AVAILABLE = False  # -> True après authentification
PROJECT_ID = 'isheero-hackathon-2026' # Mettre le nom du projet sur big query

#Pour se connecter au projet sur Big Query
if BIGQUERY_AVAILABLE:
    from google.colab import auth
    from google.cloud import bigquery
    auth.authenticate_user()
    client = bigquery.Client(project=PROJECT_ID)
    print('BigQuery connecté')


In [ ]:
# Colonnes à charger 
COLS_EVENTS = '''
    MonthYear,
    Actor1Name, Actor1CountryCode, Actor1Type1Code,
    Actor2Name, Actor2CountryCode, Actor2Type1Code,
    EventRootCode, QuadClass, GoldsteinScale,
    NumMentions, NumArticles, AvgTone,
    ActionGeo_CountryCode, ActionGeo_FullName,
    ActionGeo_Lat, ActionGeo_Long
'''

requeteSQL = f'''
    SELECT {COLS_EVENTS},
              ROW_NUMBER() OVER (
                  PARTITION BY MonthYear
                  ORDER BY RAND()
              ) as rn
        FROM `gdelt-bq.gdeltv2.events`WHERE Year >= 2025 AND (
              ActionGeo_CountryCode = 'BN'
              OR Actor1CountryCode  = 'BN'
              OR Actor2CountryCode  = 'BN'
          )
    '''


def estimate_query_cost(query: str) -> float:
    """
    Retourne le volume en GB qu'une requête va scanner.
    Ne consomme AUCUN quota BigQuery.
    """
    if not BIGQUERY_AVAILABLE:
        print('  [DRY RUN simulé - BigQuery non connecté]')
        return 0.0
    job_config = bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False
    )
    job = client.query(query, job_config=job_config)
    gb  = job.total_bytes_processed / 1e9
    pct = gb / 10
    status = 'OK' if gb < 50 else 'DANGER - reduire la requete'
    print(f'  Volume  : {gb:.3f} GB')
    print(f'  Quota   : {pct:.3f}% de votre 1 TB mensuel')
    print(f'  Statut  : {status}')
    return gb

# Test
if BIGQUERY_AVAILABLE:
    print('Estimation requete test :')
    estimate_query_cost(requeteSQL)

In [ ]:
# Lancer la requete et enregistrer dans le dossier data/raw

df_m = client.query(requeteSQL).to_dataframe()
df_m.to_parquet('data/raw/eventsBenin.parquet', index=False)

df_m.head(5)